# Business Framing

What decision the model would support, how it is made today, and what a useful result looks like, or, for an exploratory project, what it should teach and what each answer would lead to. The mode and the two anchors are settled before anyone looks at the data; the rest is drafted from what data discovery found.

**How to use this notebook.** The code cells marked *Decision* hold the answers. Edit them, then choose Run All. The last cells check the answers, write them to `ml_spec.yaml`, and update the stakeholder summary `docs/business_framing.md`. The ML definition of the problem (target, window, metric) comes later, in `03_problem_framing.ipynb`.

## Setup

Finds the project root, loads the shared spec `ml_spec.yaml`, and checks the kernel. Run this cell first.

In [1]:
import os
import sys
from pathlib import Path

# The project root is the closest folder with ml_spec.yaml. Everything runs from there.
here = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in here if (p / "ml_spec.yaml").is_file()), Path.cwd())
os.chdir(ROOT)
# The project's own code (src) comes from this folder, also in an experiment's worktree, where the editable
# install in .venv would point to the main project.
sys.path.insert(0, str(ROOT))

# Shared code from the ML project skills
sys.path.insert(0, r"/Users/ani/.claude/skills/ml-business-framing/scripts")
import ml_common
import nb_common
try:  # MLflow tracking, in the ML project skills: keeps a copy of each chart for the stage's run
    import tracking
    tracking.capture_figures()
except ImportError:
    tracking = None

SPEC_PATH = str(ROOT / "ml_spec.yaml")
spec = ml_common.migrate(ml_common.load(SPEC_PATH)) if os.path.exists(SPEC_PATH) else ml_common.template()
PATHS = ml_common.project_paths(spec, str(ROOT))
NOTEBOOK = os.path.join(PATHS["notebooks"], "01_business_framing.ipynb")
STAGE = "business"

print("Project root:", ROOT)
print("Python:", sys.executable)
if (ROOT / ".venv").is_dir() and Path(sys.prefix).resolve() != (ROOT / ".venv").resolve():
    print("Warning: this kernel is not the project's .venv. Choose 'Select Kernel' and pick the project kernel.")

Project root: /Users/ani/Projects/16_weather_prediction
Python: /Users/ani/Projects/16_weather_prediction/.venv/bin/python


In [2]:
business = {}  # filled in by the decision cells below

## Project Mode

This is a portfolio project with no business owner or KPI behind it, so it is framed as an experiment.

*mode is "standard" (a clear business goal and KPI) or "exploratory" (an experiment, framed by what it should teach and what each answer would lead to).*

In [3]:
# Decision
business.update({
    'mode': 'exploratory',
})

## What We Want to Learn

The user's main goal is to build the serving system end to end (feature store, champion/challenger gate, monitoring). That alone gives the modelling stages nothing to measure, so the learning question also asks whether the model beats persistence, which the gate and the monitoring can judge against. The target is the daily maximum temperature, one day ahead only. Each answer leads to a different action.

*answer_actions is a list of {answer, action}. Every answer should lead to a different action; if they all lead to the same one, the experiment cannot change anything.*

In [4]:
# Decision
business.update({
    'learning_question': "Can I build an end to end online inference system (feature store, champion/challenger gate, monitoring) that serves next day maximum temperature forecasts for 50 cities, with a model that beats a simple persistence forecast (tomorrow's maximum equals today's maximum)?",
    'answer_actions': [
        {
            'answer': 'The model clearly beats persistence (for example 10% or more lower error).',
            'action': 'Promote it as champion, build the full serving stack around it, and use the gate to judge later challengers.',
        },
        {
            'answer': 'The model only matches persistence.',
            'action': 'Still build the system, but serve the persistence forecast as the first champion, then try richer features (lags from nearby cities, hourly data) as challengers.',
        },
        {
            'answer': 'The model is worse than persistence.',
            'action': 'Look for leakage or a flawed setup first, and ship nothing until it is explained.',
        },
    ],
})

## Who Uses the Result

*Who reads the answer and takes the action.*

In [5]:
# Decision
business.update({
    'users': None,
})

## What to Predict

*In business terms only. The exact target, window, and columns are settled in 03_problem_framing, after data discovery.*

In [6]:
# Decision
business.update({
    'outcome': None,
})

## How the Output Is Used

*output_use is one of ranked_list, threshold_decision, probability, quantity, or other. It decides the metric family.*

In [7]:
# Decision
business.update({
    'output_use': None,
    'v1_bar_intent': None,
})

## Is ML Needed?

*ml_needed is "yes", "no", or "unsure". "no" keeps the modelling skills blocked.*

In [8]:
# Decision
business.update({
    'ml_needed': None,
    'ml_needed_reason': None,
})

## Constraints and Scope

*Limits on latency, explanation, fairness, and data access, and what this model will not do.*

In [9]:
# Decision
business.update({
    'constraints': {'latency': None, 'interpretability': None, 'fairness': None, 'data_access': None, 'other': []},
    'out_of_scope': [],
})

## Risks Found During Framing

Each problem found when the answers were checked, and how it was resolved. A problem that is still open belongs in the open questions below as well.

In [10]:
# Decision
risks = [
    {
        'problem': 'The first draft of the goal was only "build the system end to end", which gives the gate and the monitoring no quality bar.',
        'resolution': 'The learning question now includes beating a persistence forecast.',
    },
]

## Assumptions, Open Questions, and Decisions

Assumptions are things taken as true without confirmation. Each open question names the stage that needs the answer. Decisions record choices and why they were made. They are added to the shared lists in the spec; entries already there are not duplicated.

In [11]:
# Decision
assumptions = [
    'The forecast is for the next day only (horizon of one day), for the daily maximum temperature at 2 m.',
]

open_questions = []

decisions = [
    {
        'decision': 'Predict the daily maximum temperature one day ahead, not minimum or mean, and not 1 to 7 days.',
        'reason': 'The user chose it to keep V1 focused on building the system end to end.',
    },
]

## Status

`confirmed` once the business owner has agreed with the framing, `draft` until then.

In [12]:
# Decision
STATUS = 'draft'

## Write to the Spec

Checks the answers, writes them to the `business` section of `ml_spec.yaml`, and records this run.

In [13]:
assert STATUS in ("draft", "confirmed"), "STATUS must be draft or confirmed"
assert business.get("ml_needed") in (None, "yes", "no", "unsure"), 'ml_needed must be "yes", "no", or "unsure"'
assert business.get("output_use") in (None, "ranked_list", "threshold_decision", "probability", "quantity", "other")
assert business.get("mode") in (None, "standard", "exploratory"), 'mode must be "standard" or "exploratory"'
for a in business.get("answer_actions") or []:
    assert isinstance(a, dict) and a.get("answer") and a.get("action"), "each answer_actions item needs an answer and an action"

result = nb_common.write_stage(SPEC_PATH, "business", {**business, "risks": risks, "status": STATUS}, "ml-business-framing",
                               assumptions, open_questions, decisions)
nb_common.show_result(*result, STATUS)
nb_common.record_run(SPEC_PATH, STAGE, NOTEBOOK, "ml-business-framing")

Required fields still empty: business.outcome
Recommended fields still empty: business.output_use, business.v1_bar_intent, business.ml_needed


## Stakeholder Summary

Writes `docs/business_framing.md`, a short summary without code, from the spec. Edit this notebook, not that file, so the two stay the same.

In [14]:
import business_summary
path = business_summary.write(SPEC_PATH, PATHS["docs"])
print("Wrote", path)

Wrote /Users/ani/Projects/16_weather_prediction/docs/business_framing.md


## Next Step

If only the mode and the two anchors are filled in, put the data in the raw data folder, run `/ml-data-discovery`, and then come back to `/ml-business-framing` for the rest, drafted from the data. If the framing is complete and ML is needed, continue with `/ml-problem-framing`. If the question is whether a change worked (an experiment or a campaign with a control group), run `/ab-project-setup` and then `/ab-test-framing` instead. Check progress with `/ml-status`.